# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# ProbSS 5 — Learning a Markov chain from data

## What you will do

Using a local daily-rainfall series, you will turn observations into two
states, estimate transition probabilities from an earlier period, and test
predictions on later days. You will also simulate the fitted chain and look
for patterns that a first-order, time-homogeneous model misses.

This notebook is separate from the Lecture 9 notebook.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pathlib import Path

def course_data(filename):
    candidates = (
        Path("data") / filename,
        Path("master/jp/data") / filename,
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from master/jp "
        "or from the repository root."
    )


## 1. Decide what the observations and states mean

One observation is one dated rainfall amount in Christchurch. Define state 0 as dry (0 mm) and state 1 as wet (strictly positive rainfall). The threshold is fixed before fitting.

The first-order homogeneous Markov model assumes that, conditional on today's state, tomorrow's state does not depend on earlier states, and that one transition matrix applies throughout the period.


In [ ]:
rain = pd.read_csv(
    course_data("rainfallInChristchurch.csv"),
    names=["date_code", "rain_mm"],
)
rain["date"] = pd.to_datetime(
    rain["date_code"].round().astype("int64").astype(str),
    format="%Y%m%d",
    errors="raise",
)
rain["rain_mm"] = pd.to_numeric(rain["rain_mm"], errors="coerce")
rain = rain.dropna(subset=["date", "rain_mm"]).sort_values("date")
duplicate_rows = int(rain.duplicated(["date", "rain_mm"]).sum())
rain = rain.drop_duplicates(["date", "rain_mm"]).reset_index(drop=True)
rain["wet"] = (rain["rain_mm"] > 0).astype(np.int8)
calendar_gap = rain["date"].diff().dt.days
nonconsecutive_gaps = int(calendar_gap.iloc[1:].ne(1).sum())

assert rain["date"].is_monotonic_increasing
assert not rain["date"].duplicated().any()
assert set(rain["wet"].unique()).issubset({0, 1})
print(rain[["date", "rain_mm", "wet"]].head())
print("Date range:", rain["date"].min().date(), "to", rain["date"].max().date())
print("Exact duplicate rows removed:", duplicate_rows)
print("Nonconsecutive calendar gaps retained:", nonconsecutive_gaps)


## 2. Fit in time order and save a test period


In [ ]:
states = rain["wet"].to_numpy()
split = int(0.8 * len(states))
train_states = states[:split]
test_states = states[split:]

train_dates = rain.loc[: split - 1, "date"].to_numpy()
consecutive_train_pair = (
    np.diff(train_dates).astype("timedelta64[D]").astype(int) == 1
)
counts = np.zeros((2, 2), dtype=int)
for current, following in zip(
    train_states[:-1][consecutive_train_pair],
    train_states[1:][consecutive_train_pair],
):
    counts[current, following] += 1

row_totals = counts.sum(axis=1, keepdims=True)
assert (row_totals > 0).all()
P = counts / row_totals

transition_table = pd.DataFrame(
    P,
    index=["current dry", "current wet"],
    columns=["next dry", "next wet"],
)
print("Consecutive-day transition counts:\n", counts)
transition_table


In [ ]:
assert np.allclose(P.sum(axis=1), 1)
state_names = np.array(["dry", "wet"])
positive_edges = [
    (state_names[current], state_names[following], P[current, following])
    for current in range(2)
    for following in range(2)
    if P[current, following] > 0
]

# For a two-state chain, both cross-state edges give irreducibility. In an
# irreducible two-state chain, either positive self-loop gives aperiodicity.
irreducible = bool(P[0, 1] > 0 and P[1, 0] > 0)
aperiodic = bool(irreducible and (P[0, 0] > 0 or P[1, 1] > 0))

switching_mass = P[0, 1] + P[1, 0]
if switching_mass > 0:
    stationary = np.array([P[1, 0], P[0, 1]]) / switching_mass
    assert np.allclose(stationary @ P, stationary)
else:
    stationary = np.array([np.nan, np.nan])

print("Positive directed edges (from, to, probability):")
for edge in positive_edges:
    print(edge)
print("Irreducible:", irreducible)
print("Aperiodic:", aperiodic)
print("Estimated stationary distribution [dry, wet]:", stationary)
print("Training wet fraction:", train_states.mean())


Interpret each positive entry $P(i,j)$ as a directed edge from today's state $i$ to tomorrow's state $j$, weighted by its fitted transition probability. For this two-state chain, positive edges in both directions make the graph strongly connected (the chain is irreducible). A positive self-loop then makes the irreducible chain aperiodic. These checks concern the fitted model, not proof that the rainfall process is truly homogeneous or first-order Markov.

The stationary distribution is also a property of the fitted matrix. It is not a claim that the observed time series has already converged or that the weather mechanism is stationary. The same graph language is used for random walks on larger networks: states become vertices and positive transition probabilities become directed weighted edges.


## 3. Simulate from the fitted chain


In [ ]:
rng = np.random.default_rng(2026)

# Start on the last training day. For each held-out record, advance once per
# elapsed calendar day and then store the state aligned with that record.
heldout_dates = rain.loc[split:, "date"].to_numpy()
boundary_and_heldout_dates = rain.loc[split - 1 :, "date"].to_numpy()
day_gaps = np.diff(boundary_and_heldout_dates).astype("timedelta64[D]").astype(int)
assert len(day_gaps) == len(test_states)
assert np.all(day_gaps >= 1)

simulated = np.empty(len(test_states), dtype=np.int8)
current_state = int(train_states[-1])
for index, gap in enumerate(day_gaps):
    for _ in range(gap):
        current_state = rng.choice(2, p=P[current_state])
    simulated[index] = current_state

assert len(simulated) == len(test_states)
print(f"Held-out observed wet fraction: {test_states.mean():.3f}")
print(f"Simulated wet fraction:         {simulated.mean():.3f}")

n_show = min(180, len(test_states))
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.step(heldout_dates[:n_show], test_states[:n_show], where="post", label="held-out observed")
ax.step(
    heldout_dates[:n_show],
    simulated[:n_show] - 0.04,
    where="post",
    alpha=0.75,
    label="one aligned simulation (offset)",
)
ax.set(
    xlabel="date in held-out period",
    ylabel="wet indicator",
    title="Observed sequence and one fitted-chain simulation",
    yticks=[0, 1],
)
ax.legend()
plt.show()


## 4. Does dependence improve next-day prediction?


In [ ]:
evaluation_dates = rain.loc[split - 1 :, "date"].to_numpy()
consecutive_test_pair = (
    np.diff(evaluation_dates).astype("timedelta64[D]").astype(int) == 1
)
previous = states[split - 1 : -1][consecutive_test_pair]
following = states[split:][consecutive_test_pair]
markov_prob_wet = P[previous, 1]
iid_prob_wet = np.full_like(markov_prob_wet, train_states.mean(), dtype=float)

def binary_log_loss(y_true, probability):
    probability = np.clip(probability, 1e-12, 1 - 1e-12)
    return -np.mean(
        y_true * np.log(probability)
        + (1 - y_true) * np.log(1 - probability)
    )

markov_loss = binary_log_loss(following, markov_prob_wet)
iid_loss = binary_log_loss(following, iid_prob_wet)
print(f"Held-out Markov one-step log loss: {markov_loss:.4f}")
print(f"Held-out IID Bernoulli log loss:   {iid_loss:.4f}")
print("Consecutive held-out transitions evaluated:", len(following))


In [ ]:
def regular_lag_correlation(values, lag):
    x = values[:-lag]
    y = values[lag:]
    if x.std() == 0 or y.std() == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def calendar_lag_correlation(frame, lag):
    present = frame[["date", "wet"]].rename(columns={"wet": "present"})
    future = frame[["date", "wet"]].rename(columns={"wet": "future"})
    future["date"] = future["date"] - pd.Timedelta(days=lag)
    pairs = present.merge(future, on="date", how="inner")
    if pairs["present"].std() == 0 or pairs["future"].std() == 0:
        return np.nan
    return pairs["present"].corr(pairs["future"])

lags = np.arange(1, 15)
test_frame = rain.iloc[split:][["date", "wet"]]
observed_corr = np.array(
    [calendar_lag_correlation(test_frame, lag) for lag in lags]
)
simulated_corr = np.array(
    [regular_lag_correlation(simulated, lag) for lag in lags]
)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(lags, observed_corr, marker="o", label="held-out observed")
ax.plot(lags, simulated_corr, marker="s", label="fitted-chain simulation")
ax.axhline(0, color="black", linewidth=1)
ax.set(xlabel="lag in days", ylabel="correlation", title="Dependence diagnostic")
ax.legend()
ax.grid(alpha=0.2)
plt.show()


In [ ]:
criticism = pd.DataFrame(
    {
        "model choice": [
            "binary wet/dry states",
            "first-order memory",
            "one transition matrix",
            "missing or duplicated dates",
            "chronological held-out period",
            "IID baseline",
        ],
        "what it misses or tests": [
            "rainfall amount and threshold sensitivity",
            "longer wet or dry spells",
            "seasonality, climate trends, and rule changes",
            "duplicates are removed and only consecutive-day transitions are fitted",
            "future-like evaluation but not independence",
            "ignores the observed lag dependence",
        ],
    }
)
criticism


## Recap

Before you finish, make sure you can:

1. Report the fitted transition matrix, interpret each row, and draw its positive-entry directed graph. State whether the fitted two-state chain is irreducible and aperiodic, with reasons.
2. Verify the stationary calculation and explain why it is a property of the fitted matrix rather than evidence that the weather mechanism is stationary.
3. Compare the Markov and IID held-out log losses without claiming that a small difference proves the Markov model is true.
4. Use the lag plot to name one failure of the first-order homogeneous model.
5. Propose one additional state or covariate and explain what data would be needed to fit it.
